# <font color = 'red'> DEPENDENCIAS

In [16]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import MinMaxScaler

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio)
from visualization_tools import plot_interactive_chart
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> CARGA DE DATOS

In [17]:
df = pd.read_csv(get_data_path("bivariate_preprocessed_data.csv"))

In [18]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
No duplicate rows found.


# <font color = 'red'> ANÁLISIS

In [19]:
col = "Credit_History_Age"

## <font color = 'skyblue'> ANÁLISIS GENERAL

Las medianas tienen el orden esperado: Median Bad < Mediana Standard < Mediana Good.

Esto concuerda con la hipótesis de que los deudores "menos experimentados" tienden a pagar peor.

In [20]:
fig_box = px.box(df, x="Credit_Mix", y=col, title=f"Distribution of {col} by Credit Score Category")
fig_box.show()

## <font color = 'skyblue'> ANÁLISIS POR DECILES

In [21]:
continuous_variable= col
decile_col_name = continuous_variable + '_Decile'
target_col_string = "Credit_Mix" # variable dependiente con nombres string
target_col = 'Credit_Score' # variable dependiente int (para modelos)

In [22]:
analysis_summary = summarize_decile_analysis(df, continuous_variable, decile_col_name, target_col_string)

# Obtener los resultados
df_deciles = analysis_summary["df_deciles"]  # DataFrame con los deciles asignados
deciles_summary = analysis_summary["decile_summary"]  # Resumen de deciles con conteos y proporciones
display(deciles_summary)
res = check_dataframe_quality(df_deciles)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Credit_History_Age_Decile,,,,,,,,,,
0,1,84,10174,0.10174,7694,0,2480,0.756241,0.000000,0.243759
1,85,124,9974,0.09974,5428,0,4546,0.544215,0.000000,0.455785
2,125,163,9879,0.09879,5282,0,4597,0.534670,0.000000,0.465330
3,164,196,10187,0.10187,2685,1897,5605,0.263571,0.186218,0.550211
4,197,219,9804,0.09804,1234,3140,5430,0.125867,0.320277,0.553856
5,220,243,10209,0.10209,1387,3154,5668,0.135861,0.308943,0.555196
6,244,282,9905,0.09905,58,5414,4433,0.005856,0.546593,0.447552
7,283,323,10105,0.10105,0,5528,4577,0.000000,0.547056,0.452944
8,324,362,9819,0.09819,0,5455,4364,0.000000,0.555556,0.444444


No missing values found.
No infinite values found.
No duplicate rows found.


In [23]:
df[(df[continuous_variable] >= 35.665094) & (df[continuous_variable] <= 37.317021) & (df['Credit_Score'] == 0)].shape

(140, 92)

Proporción de Buenos: se observa una relación positiva entre la experiencia crediticia y la proporción de buenos deudores.

Proporción de standard: la realación entre la experiencia crediticia y la proporción de deudores es mixta: una relación positiva hasta los 240 meses de antiguedad, por encima de este umbral, la relación es negativa.

Proporción de malos: la proporción de malos y la experiencia crediticia se relacionan negativamente, de modo que a mayor experiencia crediticia menor la proporción de deudores malos.

In [24]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

<font color = 'brown'> Agrupación de deciles

Se agrupan deciles buscando una relación monótona entre las proporciones y la variable:

In [25]:
group_map = {0: "Group_1", 
             1: "Group_2", 
             2: "Group_2", 
             3: "Group_3", 
             4: "Group_4",
             5: "Group_4", 
             6: "Group_5",
             7: "Group_5",
             8: "Group_5",
             9: "Group_5"}

# Agrupar los deciles
grouped_col_name = "Grouped_" + continuous_variable

df_deciles_grouped = group_deciles(df_deciles, decile_col_name, grouped_col_name, group_map)

summary_results = summarize_grouped_deciles(df_deciles_grouped, grouped_col_name, continuous_variable, target_col_string, prefix="Decile_")
grouped_deciles_summary = summary_results['df']


# Definir el mapeo manual de los grupos a enteros
group_mapping = {
    'Group_1': 1,
    'Group_2': 2,
    'Group_3': 3,
    'Group_4': 4,
    'Group_5': 5
}

if not set(group_map.values()) == set(group_mapping.keys()):
    print("Problemas en el mapeo de grupos a enteros!")

# Usar `.map()` en lugar de `.replace()` para evitar el warning
df_deciles_grouped[grouped_col_name] = (
    df_deciles_grouped[grouped_col_name]
    .map(group_mapping)  # Mapear los valores
    .astype("Int64")      # Convertir a entero manejando NaN si existen
)

display(grouped_deciles_summary)

res = check_dataframe_quality(df_deciles_grouped)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Grouped_Credit_History_Age,,,,,,,,,,
Group_1,1,84,10174,0.10174,7694,0,2480,0.756241,0.000000,0.243759
Group_2,85,163,19853,0.19853,10710,0,9143,0.539465,0.000000,0.460535
Group_3,164,196,10187,0.10187,2685,1897,5605,0.263571,0.186218,0.550211
Group_4,197,243,20013,0.20013,2621,6294,11098,0.130965,0.314496,0.554540
Group_5,244,404,39773,0.39773,58,22193,17522,0.001458,0.557992,0.440550


No missing values found.
No infinite values found.
No duplicate rows found.


In [26]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=grouped_deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

## <font color = 'skyblue'> REGRESIONES BIVARIADAS

Como Credit_Score tiene tres categorías (Bad, Standard, Good) se pueden usar dos enfoques 
de regresión categórica: 

- Regresión Logística Multinomial → No asume orden en las categorías (como si fueran colores: rojo, azul, verde).
- Regresión Logística Ordinal → Asume que hay un orden en las categorías (Bad < Standard < Good).

Dado que hay un orden entre las categorías se utiliza Regresión Logística Ordinal:

<font color = 'gold'> Sin Agrupaciones

In [27]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable
res = check_dataframe_quality(df_)

No missing values found.
No infinite values found.
No duplicate rows found.


In [28]:
# para no generar problemas numéricos debe escalarse esta variable.
# se opta por normalizar la variable:

g(df_[[x_variable]].describe()).transpose()

,count,mean,std,min,25%,50%,75%,max
Credit_History_Age,"100,000.00",221.21,99.68,1.00,144.00,219.00,302.00,404.00


Todos los coeficientes son significativos.

Credit_History_Age_Scaled 6.4135: el coeficiente es positivo: por cada mes de experiencia adicional, la probabilidad de estar en una categoría superior de Credit_Score aumenta.

Threshold 0/1 1.7909: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 1.1008: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [29]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable

scaler = MinMaxScaler()
df_[continuous_variable + "_Scaled"] = scaler.fit_transform(df_[[x_variable]])

res = check_dataframe_quality(df_)

# Ajustar el modelo de regresión logística ordinal con la variable escalada
model_income = OrderedModel(df_[target_col], df_[continuous_variable + "_Scaled"], distr="logit")
result_income = model_income.fit(method='bfgs')

# Mostrar resumen del modelo
print(result_income.summary())

# Calcular e interpretar el Odds Ratio
res_odds = compute_odds_ratio(result_income, variable_name=continuous_variable + "_Scaled", description="Ingreso Anual Escalado")
print(res_odds["interpretation"])


No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.815496
         Iterations: 14
         Function evaluations: 15
         Gradient evaluations: 15
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -81550.
Model:                   OrderedModel   AIC:                         1.631e+05
Method:            Maximum Likelihood   BIC:                         1.631e+05
Date:                Sun, 06 Apr 2025                                         
Time:                        13:09:47                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                coef    s

<font color = 'gold'> Por Deciles

Todos los coeficientes son significativos.

Credit_History_Age_Decile 0.5488: Por cada decil adicional, la probabilidad de estar en una categoría superior de Credit_Score aumenta.

Threshold 0/1 0.7291: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 1.1126: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [30]:
df_ = df_deciles_grouped.copy()
x_variable = decile_col_name

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Ingreso Neto')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.817466
         Iterations: 13
         Function evaluations: 15
         Gradient evaluations: 15
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -81747.
Model:                   OrderedModel   AIC:                         1.635e+05
Method:            Maximum Likelihood   BIC:                         1.635e+05
Date:                Sun, 06 Apr 2025                                         
Time:                        13:09:50                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                coef    s

<font color = 'gold'> Por Agrupamientos de Deciles

Todos los coeficientes son significativos.

Grouped_Credit_History_Age 1.2221: Por cada grupo adicional, la probabilidad de estar en una categoría superior de Credit_Score aumenta.

Threshold 0/1 2.5277: Umbral que separa las categorías Bad y Standard.

Threshold 1/2 1.1941: Umbral que separa las categorías Standard y Good. Si la puntuación supera 0.7604, es más probable que 
sea Good en lugar de Standard.

In [31]:
df_ = df_deciles_grouped.copy()
x_variable = grouped_col_name

df_[target_col] = df_[target_col].astype(int)
df_[x_variable] = df_[x_variable].astype(int)

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Edad')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.787499
         Iterations: 13
         Function evaluations: 15
         Gradient evaluations: 15
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -78750.
Model:                   OrderedModel   AIC:                         1.575e+05
Method:            Maximum Likelihood   BIC:                         1.575e+05
Date:                Sun, 06 Apr 2025                                         
Time:                        13:09:52                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                 coef    

## <font color = 'skyblue'> CONCLUSIONES

### 📊 Comparación de Representaciones de `Credit_History_Age` usando Regresión Ordinal

| Representación                         | Coeficiente principal | Indicadores de ajuste                                                                 | Interpretación                                                                 |
|----------------------------------------|------------------------|----------------------------------------------------------------------------------------|--------------------------------------------------------------------------------|
| `Credit_History_Age_Scaled`           | 6.4135                | **Log-Likelihood**: -81,550<br>**AIC**: 163,101<br>**BIC**: 163,142                    | 🔹 Ajuste intermedio.<br>🔹 Relación positiva fuerte y directa con el score.<br>🔹 Escalada, por tanto conserva continuidad. |
| `Credit_History_Age_Decile`           | 0.5488                | **Log-Likelihood**: -81,747<br>**AIC**: 163,499<br>**BIC**: 163,540                    | 🔹 Ligera pérdida de ajuste.<br>🔹 Discretización útil para análisis interpretativos.<br>🔹 Coeficiente claramente positivo. |
| `Grouped_Credit_History_Age`          | 1.2221                | **Log-Likelihood**: -78,750<br>**AIC**: 157,500<br>**BIC**: 157,541                    | 🔹 Mejor ajuste.<br>🔹 Representación agrupada mejora el balance entre simplicidad y poder explicativo. |


## <font color = 'skyblue'> EXPORTACIÓN DE DATOS CON VARIABLES ADICIONALES

In [32]:
# df_deciles_grouped.to_csv("../../calibration_data/preprocessed_data.csv", index=False)
